### Libraries

In [ ]:
!pip install lightgbm

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from lightgbm import LGBMClassifier
from sklearn.metrics import recall_score, f1_score, make_scorer

# Load the cleaned dataset
df_final = pd.read_csv("dataset_transformado_ok.csv")

# Universidad Nacional de la Matanza

Specialization in Data Science

Benko Teo
Cura Diego
Riganti Valentina
Sanjuan Oriana

# **Stage: Modeling - Part 2**
*Automated Machile Learning (AutoML).*

*Purpose*: Comparing the results obtained from applying a new AutoML technique with those developed using PyCaret.

In [ ]:
# --- 1. Data preparation and split ---
TOP_15_FEATURES = [
    'VIV_DESCONOCIDO', 'psicologica', 'BA_DESCONOCIDO', 'NE_SECUNDARIO', 'SL_INFORMAL',
    'convivencia_pea_bin', 'SL_DESCONOCIDO', 'RV_parientes_convivientes', 'tratamiento_bin',
    'diagnostico_bin', 'SUMA_VIOLENCIA', 'fisica', 'RV_desconocido', 'PER_desconocido',
    'NE_OTROS'
]
target_col = 'denuncio_target'


for col in df_final.columns:
    if df_final[col].dtype == 'bool':
        df_final[col] = df_final[col].astype(int)

X = df_final[TOP_15_FEATURES].fillna(0).values
Y = df_final[target_col].values

# data split (80/20)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, stratify=Y
)

# --- 2. optimization ---

# 2.1. define model
lgbm = LGBMClassifier(random_state=42, class_weight='balanced')

# 2.2. Hyperparameter Tuning
param_grid = {
    'n_estimators': [50, 100],         
    'learning_rate': [0.05, 0.1],    
    'num_leaves': [10, 20],           
}

# 2.3. Optimization Strategy: Sensitivity-Driven Scoring.
recall_scorer = make_scorer(recall_score)

# 2.4.  GridSearchCV
grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    scoring=recall_scorer,
    cv=5, # 5-fold Cross-Validation
    n_jobs=-1,
    verbose=2
)

print("--- Starting GridSearchCV (Simulating AutoML Tuning) ---")
grid_search.fit(X_train, Y_train)

# --- 3. final assessment ---

# 3.1. Obtaining the Best Model
best_lgbm = grid_search.best_estimator_

# 3.2. Prediction on the Test Set
Y_pred = best_lgbm.predict(X_test)

# 3.3. metrics
final_recall = recall_score(Y_test, Y_pred)
final_f1 = f1_score(Y_test, Y_pred)

print("\n--- Final Results: Optimized LightGBM ---")
print(f"Best Parameter Found: {grid_search.best_params_}")
print(f"Recall (class 1, warning): {final_recall:.4f}")
print(f"F1-Score (Balance): {final_f1:.4f}")

--- Iniciando GridSearchCV (Simulando Tuning AutoML) ---
Fitting 5 folds for each of 8 candidates, totalling 40 fits
[LightGBM] [Info] Number of positive: 76, number of negative: 112
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000075 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 30
[LightGBM] [Info] Number of data points in the train set: 188, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# Results

Final Results: Optimized LightGBM Performance
The LightGBM Classifier, despite being a powerful tree-based model, failed to outperform the linear model (Ridge) in the crucial alert metric (Recall).

* Recall (Class 1): 0.5263 (52.6%) vs. 0.6842 (68.4%). INFERIOR PERFORMANCE. LightGBM omits more False Negatives.

* F1-Score: 0.5263 vs. 0.6500. INFERIOR PERFORMANCE. Poorer balance between sensitivity and precision

**Final decision**

---


*Model Selection: Ridge Classifier*. Despite its simplicity, its linear nature was more effective in detecting "registration failure" patterns (such as $\mathbf{VIV\_DESCONOCIDO}$) which are highly correlated and linear. *LightGBM*, by searching for complex interactions, could not outperform the weight of these features.Furthermore, a Recall of 0.5263 is insufficient for an early warning system, as it would mean omitting nearly half ($\approx 47\%$) of the real risk cases. The Recall of 0.6842 from the Ridge Classifier is superior.In conclusion, although advanced non-linear models (LightGBM) were tested, the Ridge Classifier proved to be the most robust, offering the best balance (Recall/F1-Score), which is essential for the system's implementability.